# Notebook 03 — From Embeddings to Queries, Keys and Values (Q, K, V)

## Goal

In this notebook you will learn:

- How token embeddings become Queries, Keys and Values.
- Why Q, K and V all start from the same embedding.
- Why they become different vectors.
- How `nn.Linear` performs the required matrix multiplication.
- The tensor shapes involved in every step.

Run every cell in order and read the explanation immediately after each code cell.


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print(torch.__version__)

2.12.0+cu130


### Explanation

- `torch` is the main tensor library.
- `torch.nn` contains neural network layers.
- `torch.manual_seed(42)` makes the random numbers reproducible.


In [2]:
embeddings = torch.tensor([
    [0.5, 1.2, -0.3, 2.0],
    [-1.0, 0.7, 0.4, -0.2],
    [1.5, -0.8, 0.9, 0.3]
], dtype=torch.float32)

print("Embeddings:")
print(embeddings)
print("Shape:", embeddings.shape)

Embeddings:
tensor([[ 0.5000,  1.2000, -0.3000,  2.0000],
        [-1.0000,  0.7000,  0.4000, -0.2000],
        [ 1.5000, -0.8000,  0.9000,  0.3000]])
Shape: torch.Size([3, 4])


### Explanation

Each row represents one token embedding.

Shape `(3, 4)` means:

- 3 tokens
- 4-dimensional embedding for each token

In a real GPT-2 model this would typically be `(sequence_length, 768)`.


In [3]:
d_model = 4
d_k = 4

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

print("W_Q shape:", W_Q.shape)
print("W_K shape:", W_K.shape)
print("W_V shape:", W_V.shape)

W_Q shape: torch.Size([4, 4])
W_K shape: torch.Size([4, 4])
W_V shape: torch.Size([4, 4])


### Explanation

These matrices are **learnable parameters**.

The transformer learns three different matrices:

- `W_Q`
- `W_K`
- `W_V`

Their job is to project the same embedding into three different spaces.


In [4]:
Q = embeddings @ W_Q
K = embeddings @ W_K
V = embeddings @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([3, 4])
K shape: torch.Size([3, 4])
V shape: torch.Size([3, 4])


### Explanation

This is the most important computation in today's notebook.

Mathematically:

- `Q = E @ W_Q`
- `K = E @ W_K`
- `V = E @ W_V`

The input embedding is identical, but each weight matrix is different, so the outputs become different.


In [5]:
print("Embedding (token 1):")
print(embeddings[0])

print("\nQuery:")
print(Q[0])

print("\nKey:")
print(K[0])

print("\nValue:")
print(V[0])

Embedding (token 1):
tensor([ 0.5000,  1.2000, -0.3000,  2.0000])

Query:
tensor([ 0.5474, -2.3513, -1.0213, -1.0324])

Key:
tensor([-0.9356,  0.9088,  0.0260,  3.5558])

Value:
tensor([-2.3907,  3.0018, -1.9502, -0.8663])


### Explanation

Notice that one embedding produces three different vectors.

Nothing is split.

The embedding is transformed three independent times.


In [6]:
linear_q = nn.Linear(d_model, d_k, bias=False)
linear_k = nn.Linear(d_model, d_k, bias=False)
linear_v = nn.Linear(d_model, d_k, bias=False)

Q2 = linear_q(embeddings)
K2 = linear_k(embeddings)
V2 = linear_v(embeddings)

print(Q2.shape)
print(K2.shape)
print(V2.shape)

torch.Size([3, 4])
torch.Size([3, 4])
torch.Size([3, 4])


### Explanation

This is how transformers are implemented in PyTorch.

Internally,

`nn.Linear(x)` performs matrix multiplication with a learnable weight matrix.

Conceptually:

`Q = x @ W_Q`


In [7]:
print("Original Q:")
print(Q)

W_Q_new = torch.randn(d_model, d_k)
Q_new = embeddings @ W_Q_new

print("\nNew Q after changing W_Q:")
print(Q_new)

Original Q:
tensor([[ 0.5474, -2.3513, -1.0213, -1.0324],
        [-1.6073, -1.5801, -0.9341,  0.2683],
        [ 1.4524,  4.5346,  0.8016, -2.9091]])

New Q after changing W_Q:
tensor([[ 0.0995,  3.2644,  1.5486, -1.1866],
        [ 2.8310, -0.0333,  0.1244, -0.2116],
        [-4.3642,  1.3248,  0.6872,  0.4726]])


### Explanation

Changing only `W_Q` changes the Queries immediately.

This demonstrates that the learned projection matrix determines what each token is *looking for*.


# Notebook Summary

You learned that:

- Every token starts with one embedding.
- The embedding is projected three times.
- Q, K and V are created using different learned matrices.
- `nn.Linear` is the PyTorch implementation of these projections.
- These projections are the inputs to the attention mechanism.

## Self-check

1. Why are Q, K and V different if the embedding is the same?
2. What is the equation for computing Q?
3. Why do transformers need three projection matrices instead of one?
4. What shape does Q have if the embedding shape is `(5, 768)` and `d_k = 64`?
